# Relative Lift and MDE for Ratio Metrics (Delta Method)

This notebook demonstrates how to compute **relative (percent) lift** and **minimum detectable effects (MDE)** for ratio metrics using the proper outer delta method.

## Background

For ratio metrics such as conversion rate (`clicks / impressions`) or average revenue per user (`revenue / users`), the naive approach of dividing the absolute effect by the control mean **underestimates variance** because it treats the denominator as a fixed constant.  The correct approach uses an *outer delta method* that propagates uncertainty from both the treatment effect and the control mean.

The sections below cover:
1. Data generation for a ratio metric experiment
2. Manual vs proper relative lift computation
3. Comparison of naive SE vs delta-method SE
4. MDE calculation for relative effects
5. Full `AnalysisPlan` with `RatioMetric` and `relative_effect=True`

In [ ]:
import numpy as np
import pandas as pd
from copy import deepcopy

from cluster_experiments import (
    AnalysisPlan,
    DeltaMethodAnalysis,
    DeltaMethodLiftTransformer,
    NormalPowerAnalysis,
)
from cluster_experiments.random_splitter import ClusteredSplitter

np.random.seed(42)

## 1. Data Generation

We generate a cluster-level dataset (one row per user) with:
- `scale`: number of sessions per user
- `target`: number of conversions per user (binomial draw)
- Treatment adds a 5% absolute lift to the conversion rate

In [ ]:
def generate_data(
    n_users: int = 5_000,
    treatment_effect: float = 0.05,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    user_ids = np.arange(n_users)
    treatment_flag = rng.integers(0, 2, size=n_users)
    scale = rng.integers(5, 20, size=n_users).astype(float)
    base_rate = np.clip(0.30 + rng.normal(0, 0.05, size=n_users), 0.05, 0.95)
    rate = np.clip(base_rate + treatment_effect * treatment_flag, 0, 1)
    target = rng.binomial(scale.astype(int), rate).astype(float)
    return pd.DataFrame({
        "user": user_ids,
        "treatment": np.where(treatment_flag == 0, "A", "B"),
        "target": target,
        "scale": scale,
        "pre_rate": base_rate * scale + rng.normal(0, 0.1, size=n_users),
    })

df = generate_data(n_users=10_000, treatment_effect=0.05)
print(df.head())
print(f"\nControl ratio mean: {df[df.treatment=='A']['target'].sum() / df[df.treatment=='A']['scale'].sum():.4f}")
print(f"Treatment ratio mean: {df[df.treatment=='B']['target'].sum() / df[df.treatment=='B']['scale'].sum():.4f}")

## 2. Absolute vs Relative Effect: Manual Comparison

First, let's compute the absolute effect via `DeltaMethodAnalysis` and the naive relative lift (absolute / control mean), then compare with the proper double-delta relative lift.

In [ ]:
analyser_abs = DeltaMethodAnalysis(
    cluster_cols=["user"],
    scale_col="scale",
    target_col="target",
)
analyser_rel = DeltaMethodAnalysis(
    cluster_cols=["user"],
    scale_col="scale",
    target_col="target",
    relative_effect=True,
)

# Control mean for naive computation
agg = df.groupby(["user", "treatment"], as_index=False).agg({"target": "sum", "scale": "sum"})
ctrl_mean = agg[agg["treatment"] == "A"]["target"].sum() / agg[agg["treatment"] == "A"]["scale"].sum()

abs_effect = analyser_abs.get_point_estimate(df)
abs_se = analyser_abs.get_standard_error(df)

rel_effect = analyser_rel.get_point_estimate(df)
rel_se = analyser_rel.get_standard_error(df)
rel_pvalue = analyser_rel.get_pvalue(df)

naive_rel = abs_effect / ctrl_mean
naive_se = abs_se / ctrl_mean

print(f"Control mean:           {ctrl_mean:.6f}")
print()
print(f"Absolute ATE:           {abs_effect:.6f}")
print(f"Absolute SE:            {abs_se:.6f}")
print()
print(f"Relative lift (naive):  {naive_rel:.6f}  (= {naive_rel*100:.2f}%)")
print(f"Naive SE:               {naive_se:.6f}")
print()
print(f"Relative lift (proper): {rel_effect:.6f}  (= {rel_effect*100:.2f}%)")
print(f"Proper SE (delta):      {rel_se:.6f}")
print(f"p-value (relative):     {rel_pvalue:.6f}")
print()
print(f"SE inflation from delta method: {(rel_se / naive_se - 1) * 100:.2f}%")

## 3. Using DeltaMethodLiftTransformer Directly

The `DeltaMethodLiftTransformer` can also be used standalone when you already have the delta-method statistics (e.g. from a custom analysis pipeline).

In [ ]:
# Suppose we have these summary statistics from the delta method:
mean_diff = abs_effect           # absolute ATE
var_abs = abs_se ** 2            # Var(ATE) = treat_var + ctrl_var

# We also need ctrl_mean and ctrl_var separately.
# Here we replicate the internals for illustration:
ctrl_agg = agg[agg["treatment"] == "A"]
treat_agg = agg[agg["treatment"] == "B"]

def ratio_variance_simple(sub_df):
    target = sub_df["target"].values
    scale = sub_df["scale"].values
    t_mean, s_mean = target.mean(), scale.mean()
    var_t = target.var(ddof=0)
    var_s = scale.var(ddof=0)
    cov_ts = np.cov(target, scale, ddof=0)[0, 1]
    g_t, g_s = 1 / s_mean, -t_mean / s_mean**2
    return (g_t**2 * var_t + g_s**2 * var_s + 2 * g_t * g_s * cov_ts) / len(target)

ctrl_var = ratio_variance_simple(ctrl_agg)
treat_var = ratio_variance_simple(treat_agg)

transformer = DeltaMethodLiftTransformer(treatment_col="treatment")
transformer.fit(
    mean_diff=mean_diff,
    var_abs=var_abs,
    ctrl_mean=ctrl_mean,
    ctrl_var=ctrl_var,
)

print("DeltaMethodLiftTransformer results:")
print(f"  params:  {transformer.params}")
print(f"  bse:     {transformer.bse}")
print(f"  pvalues: {transformer.pvalues}")
print(f"  conf_int (95%):")
print(transformer.conf_int(0.05))

## 4. Relative MDE via DeltaMethodLiftTransformer

The static method `DeltaMethodLiftTransformer.relative_mde` solves the quadratic power equation for the minimum detectable relative lift. Compare it with the naive MDE (linear approximation).

In [ ]:
from scipy.stats import norm as scipy_norm

alpha = 0.05
power = 0.80

# Proper relative MDE (double-delta, quadratic solution)
proper_mde = DeltaMethodLiftTransformer.relative_mde(
    alpha=alpha,
    power=power,
    ctrl_mean=ctrl_mean,
    ctrl_var=ctrl_var,
    treat_var=treat_var,
)

# Naive relative MDE (divide absolute MDE by ctrl_mean)
z_a, z_b = scipy_norm.ppf(1 - alpha / 2), scipy_norm.ppf(power)
naive_mde = (z_a + z_b) * abs_se / ctrl_mean

print(f"alpha = {alpha}, power = {power}")
print()
print(f"Naive MDE  (abs_se / ctrl_mean * z): {naive_mde*100:.3f}%")
print(f"Proper MDE (double-delta quadratic): {proper_mde*100:.3f}%")
print(f"MDE inflation: {(proper_mde / naive_mde - 1) * 100:.2f}%")

## 5. Power Analysis with NormalPowerAnalysis

`NormalPowerAnalysis` with `DeltaMethodAnalysis(relative_effect=True)` computes the MDE in relative (percent) units through simulation of the standard error.

In [ ]:
splitter = ClusteredSplitter(cluster_cols=["user"])

pw_absolute = NormalPowerAnalysis(
    analysis=DeltaMethodAnalysis(
        cluster_cols=["user"], scale_col="scale", target_col="target"
    ),
    splitter=splitter,
)

pw_relative = NormalPowerAnalysis(
    analysis=DeltaMethodAnalysis(
        cluster_cols=["user"], scale_col="scale", target_col="target",
        relative_effect=True,
    ),
    splitter=splitter,
)

df_no_treatment = df.drop(columns=["treatment"])

mde_abs = pw_absolute.mde(df_no_treatment, power=0.8, n_simulations=50)
mde_rel = pw_relative.mde(df_no_treatment, power=0.8, n_simulations=50)

print(f"Absolute MDE: {mde_abs:.6f}")
print(f"Relative MDE: {mde_rel*100:.3f}%")
print(f"Ratio (should be ~1/ctrl_mean): {mde_abs / mde_rel:.4f}  vs ctrl_mean: {ctrl_mean:.4f}")

## 6. AnalysisPlan with RatioMetric and relative_effect=True

The `AnalysisPlan` workflow supports relative effects for ratio metrics by passing `relative_effect=True` in `analysis_config`.

In [ ]:
abs_plan_config = {
    "metrics": [
        {"alias": "conversion_rate", "name": "target", "scale_name": "scale"},
    ],
    "variants": [
        {"name": "A", "is_control": True},
        {"name": "B", "is_control": False},
    ],
    "analysis_type": "delta",
    "variant_col": "treatment",
    "analysis_config": {"cluster_cols": ["user"]},
}

rel_plan_config = deepcopy(abs_plan_config)
rel_plan_config["analysis_config"] = {
    "cluster_cols": ["user"],
    "relative_effect": True,
}

abs_plan = AnalysisPlan.from_metrics_dict(abs_plan_config)
rel_plan = AnalysisPlan.from_metrics_dict(rel_plan_config)

abs_results = abs_plan.analyze(df)
rel_results = rel_plan.analyze(df)

print("Absolute AnalysisPlan results:")
print(abs_results.to_dataframe()[["metric_alias", "ate", "p_value", "std_error"]])
print()
print("Relative AnalysisPlan results:")
print(rel_results.to_dataframe()[["metric_alias", "ate", "p_value", "std_error"]])
print()
print(f"Relative ATE == Absolute ATE / ctrl_mean: {rel_results.ate[0]:.6f} == {abs_results.ate[0] / ctrl_mean:.6f}")

## 7. With CUPED Covariates

The relative effect also works with CUPED variance reduction.  Covariates require pre-aggregated cluster-level data.

In [ ]:
# Aggregate to cluster level (required when covariates are used)
df_agg = df.groupby(["user", "treatment"], as_index=False).agg(
    {"target": "sum", "scale": "sum", "pre_rate": "sum"}
)

analyser_cuped = DeltaMethodAnalysis(
    cluster_cols=["user"],
    scale_col="scale",
    target_col="target",
    covariates=["pre_rate"],
    relative_effect=True,
)

analyser_no_cuped = DeltaMethodAnalysis(
    cluster_cols=["user"],
    scale_col="scale",
    target_col="target",
    relative_effect=True,
)

rel_cuped = analyser_cuped.get_inference_results(df_agg, alpha=0.05)
rel_no_cuped = analyser_no_cuped.get_inference_results(df, alpha=0.05)

print("Without CUPED:")
print(f"  Relative ATE = {rel_no_cuped.ate*100:.3f}%,  SE = {rel_no_cuped.std_error:.6f}")
print()
print("With CUPED (pre_rate covariate):")
print(f"  Relative ATE = {rel_cuped.ate*100:.3f}%,  SE = {rel_cuped.std_error:.6f}")
print()
print(f"SE reduction from CUPED: {(1 - rel_cuped.std_error / rel_no_cuped.std_error) * 100:.1f}%")